In [ ]:
import cafo_iowa.db.session as s
import logging
import geopandas as gpd
import pandas as pd
import os
import yaml
import numpy as np
from munch import munchify
import cafo_iowa.db.models as m
import json
from shapely import unary_union
import json
from cafo_iowa.utils.strata import (
    stratified_sample,
    plot_histograms,
    check_and_save_batch,
)

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)

session = s.get_session()
engine = session.bind

In [ ]:
data = gpd.read_postgis(
    f"""
    WITH
        permit_tiles AS (
            SELECT
                naip_qt_id,
                SUM(animal_units) as animal_units,
                SUM(swine_animal_units) as swine_animal_units,
                COUNT(*) as n_facilities,
                ARRAY_AGG(facilityid) as facilityid_list
            FROM processed.{m.Permits.__tablename__}
            WHERE swine_animal_units > 0
            GROUP BY naip_qt_id
        ),
        naip21_qt AS (
            SELECT
                id as naip_qt_id,
                geometry,
                is_urban,
                urban_area
            FROM processed.{m.Naip21QT.__tablename__}
        ),
        labelled_tiles AS (
            SELECT
                jsonb_array_elements_text(naip_qt_ids) AS naip_qt_id,
                id as batch_nr,
                batch_name
            FROM processed.label_batches
        )
    SELECT
        n.*,
        COALESCE(p.animal_units, 0) as animal_units,
        COALESCE(p.swine_animal_units, 0) as swine_animal_units,
        COALESCE(p.n_facilities, 0) as n_facilities,
        p.facilityid_list,
        COALESCE(l.batch_name, 'unlabelled') as batch_name,
        l.batch_nr,
        CASE
            WHEN p.n_facilities > 0 THEN p.animal_units / p.n_facilities
            ELSE 0
        END as animal_units_per_permit,
        CASE
            WHEN p.n_facilities > 0 THEN p.swine_animal_units / p.n_facilities
            ELSE 0
        END as swine_animal_units_per_permit,
        CASE
            WHEN COALESCE(p.n_facilities, 0) = 0 THEN '0'
            WHEN p.animal_units / p.n_facilities BETWEEN 0 AND 399 THEN '1-399'
            WHEN p.animal_units / p.n_facilities BETWEEN 400 AND 499 THEN '400-499'
            WHEN p.animal_units / p.n_facilities BETWEEN 500 AND 699 THEN '500-699'
            WHEN p.animal_units / p.n_facilities BETWEEN 700 AND 899 THEN '700-899'
            WHEN p.animal_units / p.n_facilities BETWEEN 900 AND 999 THEN '900-999'
            WHEN p.animal_units / p.n_facilities BETWEEN 1000 AND 1199 THEN '1000-1199'
            WHEN p.animal_units / p.n_facilities BETWEEN 1200 AND 1499 THEN '1200-1499'
            WHEN p.animal_units / p.n_facilities BETWEEN 1500 AND 1899 THEN '1500-1899'
            WHEN p.animal_units / p.n_facilities BETWEEN 1900 AND 1999 THEN '1900-1999'
            WHEN p.animal_units / p.n_facilities BETWEEN 2000 AND 2499 THEN '2000-2499'
            WHEN p.animal_units / p.n_facilities >= 2500 THEN '2500+'
        END as animal_units_per_permit_q
    FROM naip21_qt n
    LEFT JOIN permit_tiles p USING (naip_qt_id)
    LEFT JOIN labelled_tiles l ON n.naip_qt_id = l.naip_qt_id
    """,
    s.get_engine(),
    geom_col="geometry",
)

In [ ]:
# check number of tiles per batch
print(data.groupby("batch_name").size())

In [ ]:
plot_histograms(
    data,
    data[data["batch_name"] == "IA_data_labeling_batch6"],
    "IA_data_labeling_batch6",
)